# CareBot AI - train the skin-image CNN on Colab

Trains the 12-class MobileNetV2 classifier that
`backend/app/services/image_service.py` auto-loads from `models/image_model/`.

**Before you start:** set the runtime to a GPU --
`Runtime > Change runtime type > T4 GPU`. On CPU this takes hours; on a T4 it is
roughly 20-30 minutes.

Run the cells in order. Two of them need input from you:
step 2 (your `kaggle.json`) and step 5 (the two training scripts from the repo).


## 1. Check the GPU and pin TensorFlow

We install the same TensorFlow version the backend runs (2.21.0) so the saved
`.keras` file is guaranteed to load locally.

**Colab will show a 'Restart session' button after this cell** - click it, then
carry on from cell 2. You do not need to re-run this one.


In [ ]:
!nvidia-smi -L || echo 'NO GPU - set Runtime > Change runtime type > T4 GPU'
!pip install -q 'tensorflow==2.21.0'
print('Now click RESTART SESSION above, then continue from step 2.')


In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus, 'No GPU visible - Runtime > Change runtime type > T4 GPU, then rerun.'


## 2. Kaggle credentials

Get `kaggle.json` from <https://www.kaggle.com/settings> -> API -> **Create New Token**.
Run the cell, then pick that file in the upload dialog.

You must also open each dataset page once and accept its terms, or the download
returns 403:

* <https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000>
* <https://www.kaggle.com/datasets/shubhamgoel27/dermnet>


In [ ]:
import os, json
from google.colab import files

print('Select your kaggle.json ...')
up = files.upload()
assert 'kaggle.json' in up, 'Expected a file named kaggle.json'

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'wb') as f:
    f.write(up['kaggle.json'])
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

!pip install -q kaggle
print('kaggle configured as:', json.loads(up['kaggle.json'])['username'])


## 3. Download both datasets

~5GB (HAM10000) + ~2GB (DermNet). Takes a few minutes on Colab's connection.


In [ ]:
!mkdir -p /content/data/ham /content/data/dermnet
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data/ham --unzip
!kaggle datasets download -d shubhamgoel27/dermnet -p /content/data/dermnet --unzip
!df -h /content | tail -1


## 4. Flatten any nested zips and sanity-check the layout

The HAM10000 release sometimes ships its image folders as inner `.zip` files,
so unpack anything left over before building the manifest.


In [ ]:
import glob, zipfile, os

for z in glob.glob('/content/data/**/*.zip', recursive=True):
    print('unzipping', z)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(os.path.dirname(z))
    os.remove(z)

print('\nHAM10000 top level:')
print(sorted(os.listdir('/content/data/ham'))[:10])
print('  jpgs found:', len(glob.glob('/content/data/ham/**/*.jpg', recursive=True)))
print('  metadata csv:', os.path.exists('/content/data/ham/HAM10000_metadata.csv'))

print('\nDermNet top level:')
print(sorted(os.listdir('/content/data/dermnet'))[:10])
print('  jpgs found:', len(glob.glob('/content/data/dermnet/**/*.jpg', recursive=True)))


## 5. Upload the two training scripts

These live in your local repo and are not in the public GitHub copy, so upload
them here. Select **both** files in the dialog:

* `scripts/prepare_skin_dataset.py`
* `scripts/train_image_model.py`

They go into `/content/scripts/`, which makes the scripts' own default paths
(`/content/datasets/...`, `/content/models/image_model`) resolve correctly.


In [ ]:
import os
from google.colab import files

os.makedirs('/content/scripts', exist_ok=True)
print('Select prepare_skin_dataset.py AND train_image_model.py ...')
up = files.upload()

for name, data in up.items():
    with open(os.path.join('/content/scripts', name), 'wb') as f:
        f.write(data)
    print('saved', name, len(data), 'bytes')

for required in ('prepare_skin_dataset.py', 'train_image_model.py'):
    assert os.path.exists('/content/scripts/' + required), 'missing ' + required
print('both scripts present')


## 6. Build the training manifest

Merges the two datasets into one 12-class label space and records a `group_id`
per row so repeat photographs of the same lesion cannot straddle the splits.

Read the printed class table: the imbalance ratio tells you what to expect.


In [ ]:
!python /content/scripts/prepare_skin_dataset.py \
    --ham /content/data/ham \
    --dermnet /content/data/dermnet


### Optional: add healthy-skin images

Neither dataset contains normal skin, so by default the model assigns *some*
condition to every photo. If you have your own healthy-skin photos, upload them
to a folder and re-run step 6 with `--normal /content/data/normal` added.


## 7. Train

Phase 1 trains the head on a frozen MobileNetV2; phase 2 unfreezes the top 40
layers at a low learning rate, which is where most of the real accuracy comes from.

`--batch 64` suits a T4. Drop to 32 if you hit an out-of-memory error.

Expect ~20-30 minutes. Keep the browser tab alive or Colab may reclaim the runtime.


In [ ]:
!python /content/scripts/train_image_model.py \
    --batch 64 \
    --epochs 10 \
    --finetune-epochs 8


## 8. Read the results properly

**Judge this model on macro recall, not accuracy.** Melanocytic Nevus alone is
roughly two-thirds of HAM10000, so a model that always answers with the majority
class scores ~67% accuracy while being useless.

Pay particular attention to recall on **Melanoma**, **Basal Cell Carcinoma** and
**Actinic Keratosis** - those are the classes where a miss actually matters.


In [ ]:
import json

m = json.load(open('/content/models/image_model/model_metrics.json'))
print('accuracy      %.4f' % m['accuracy'])
print('macro recall  %.4f' % m['macro_recall'])
print('macro F1      %.4f' % m['macro_f1'])
print('train/val/test: %d / %d / %d' % (m['n_train'], m['n_val'], m['n_test']))
print()
print('%-30s %8s %8s %8s' % ('class', 'prec', 'recall', 'support'))
print('-' * 58)
for label in m['labels']:
    s = m['per_class'][label]
    print('%-30s %8.2f %8.2f %8d' % (label, s['precision'], s['recall'], s['support']))


## 9. Download the trained model

Produces `image_model.zip`. Unzip it into your repo so you end up with:

```
models/image_model/model.keras
models/image_model/labels.json
models/image_model/model_metrics.json
```

Then restart the backend - it auto-loads the CNN and `is_placeholder_model`
flips to `false`. No code change needed.


In [ ]:
import os, shutil
from google.colab import files

shutil.make_archive('/content/image_model', 'zip', '/content/models/image_model')
print('zip size: %.1f MB' % (os.path.getsize('/content/image_model.zip') / 1e6))
files.download('/content/image_model.zip')


---

### Reminder

This model is trained on public research datasets and is **not** clinically
validated. It is a screening aid only, and a reassuring result is not evidence
that a lesion is harmless.
